The New York City Motor Vehicle Collisions crash table contains police-reported traffic accidents (MV-104AN). Reporting is required for collisions involving injury, death, or $1,000+ in damage. The dataset is preliminary and may be updated as forms are amended; verify current statistics on the NYPD Motor Vehicle Collisions page or Vision Zero View.

- Event & Spatial Identification: `COLLISION_ID`, `CRASH DATE` (YYYY-MM-DD), `CRASH TIME` (HH:MM), `BOROUGH`, `ZIP CODE`, `LATITUDE` / `LONGITUDE`, `LOCATION` (lat,lon pair).
- Street Registry: `ON STREET NAME`, `CROSS STREET NAME`, `OFF STREET NAME` (one used depending on location).
- Casualties: `NUMBER OF PERSONS/PEDESTRIANS/CYCLIST/MOTORIST INJURED` and corresponding `... KILLED`.
- Contributing Factors: `CONTRIBUTING FACTOR VEHICLE 1` → `5` (cause per vehicle).
- Vehicle Classification: `VEHICLE TYPE CODE 1` → `5` (e.g., sedan, SUV, truck, bicycle).

Background:
- 1998: TrafficStat launched to analyze crash data using CompStat-style methods.
- 1999: TAMS standardized manual MV-104AN reporting across precincts.
- 2016: FORMS replaced TAMS, enabling electronic MV-104AN submission and improved analysis.

Notes: report required for injury/death or $1,000+ damage; dataset may be updated by NYPD. 

Reference: https://catalog.data.gov/dataset/motor-vehicle-collisions-crashes
                

In [ ]:
%pip install pandas matplotlib numpy seaborn geopy

In [ ]:
import importlib.util  
import pandas as pd
import numpy as np
import geopy

geopy installed: True


In [5]:
# Load dataset
file_path = './dataset/'
file = file_path+'Motor_Vehicle_Collisions_-_Crashes.csv'
data = pd.read_csv(file)

C:\Users\Ross\AppData\Local\Temp\ipykernel_8544\3456463887.py:4: DtypeWarning: Columns (0: ZIP CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file)


In [ ]:
# EDA
pd.options.display.float_format = '{:.2f}'.format #limits decimal place to 2 for all columns

#print all columns
print(data.columns.tolist())

#Sum all na values
print(f"COLUMN DATA: {data.isna().sum()}")
data.describe()

In [4]:
# Cleans vehicle types and maps each value
def group_vehicle_types(val):
    if pd.isna(val):
        return np.nan
    
    text = str(val).strip().upper()
    if not text or text in {
        'NAN', 'NONE', 'NULL', 'UNK', 'UNKNOWN', 'UNKNOWN VEHICLE', 'UNKNOWN VE', 'UNSPECIFIED'
    }:
        return np.nan

    if any(k in text for k in ['AMBU', 'FIRE', 'NYPD', 'FDNY', 'EMS', 'POLI', 'POLICE']):
        return 'EMERGENCY VEHICLE'
    elif any(k in text for k in ['BIKE', 'BICY', 'SCOOT', 'MOPED', 'MOTOR', 'CYCL']):
        return 'BICYCLE/SCOOTER'
    elif any(k in text for k in ['SEDAN', 'PASSENGER', 'CAR', 'CONVERT', 'CHEV', 'FORD', 'TOYOT']):
        return 'PASSENGER CAR'
    elif any(k in text for k in ['SUV', 'WAGON', 'SUBURBAN', 'JEEP']):
        return 'SUV/STATION WAGON'
    elif any(k in text for k in ['TRUCK', 'VAN', 'PICK', 'COMM', 'TRACTOR', 'TRACKTOR', 'DELIV', 'BOX', 'FLAT']):
        return 'COMMERCIAL/TRUCK'
    elif 'BUS' in text or 'MTA' in text:
        return 'BUS'
    return 'OTHER/UNCOMMON'

In [5]:
import re
# street cleaning function
def clean_street_name(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if not text or text.upper() in {'NAN', 'NONE', 'NULL'}:
        return np.nan

    text = re.sub(r'^[\'"]+|[\'"]+$', '', text)
    text = re.sub(r'\s{2,}', ' ', text)

    return text

In [6]:
# Column Formatting
data.columns = (
    data.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_',regex=False)
    .str.replace(r'[^a-z0-9_]', '', regex=True)    
)
# contains float val, convert to int64
data['number_of_persons_injured'] = data['number_of_persons_injured'].fillna(0).astype(int)
data['number_of_persons_killed'] = data['number_of_persons_killed'].fillna(0).astype(int)

# Strip leading decimals
data['zip_code'] = data['zip_code'].astype(str).str.split('.').str[0]
data['zip_code'] = data['zip_code'].replace('nan', np.nan)

# Apply street cleaning now that columns have been normalized
street_cols = ['on_street_name', 'cross_street_name', 'off_street_name']
for col in street_cols:
    if col in data.columns:
        data[col] = data[col].apply(clean_street_name)

In [7]:
# Format vehicle types consistently for all vehicle_type_code columns
vehicle_cols = [col for col in data.columns if col.startswith('vehicle_type_code')]
unknown_pattern = r'^(NAN|NONE|NULL|UNK|UNKNOWN|UNKNOWN VEHICLE|UNKNOWN VE|UNSPECIFIED)?$'
for col in vehicle_cols:
    data[col] = (
        data[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .replace(unknown_pattern, np.nan, regex=True)
        .apply(group_vehicle_types)
    )

In [8]:
# combine data and time columns
data['crash_datetime'] = pd.to_datetime(data['crash_date'].astype(str).str.split().str[0] + ' ' + data['crash_time'],errors='coerce')
data = data.drop(columns=['crash_date', 'crash_time'])

# Standardize Street columns
street_cols = ['on_street_name', 'cross_street_name', 'off_street_name']
for col in street_cols:
    data[col] = data[col].astype(str).str.strip().str.upper()
    data[col] = data[col].replace(['NAN', 'NONE', ''], 'UNSPECIFIED')


In [9]:
# Create new column to take best street info avail
data['primary_street'] = data['on_street_name'].fillna(data['off_street_name'])

In [10]:
# Format 0.0 in both latitude and longitude to NaN
data['latitude'] = data['latitude'].replace(0.0, np.nan)
data['longitude'] = data['longitude'].replace(0.0, np.nan)

# Hardcode min anx max lat/long for NYC area
lat_min, lat_max = 40.49, 40.9276
lon_min, lon_max = -74.27, -73.68

# Replace Outliers Latitudes with NaN
data.loc[(data['latitude'] < lat_min) | (data['latitude'] > lat_max), 'latitude'] = np.nan

# Replace Outliers Longitudes with NaN
data.loc[(data['longitude'] < lon_min) | (data['longitude'] > lon_max), 'longitude'] = np.nan

In [ ]:
# This part is computationally expensive as it maps entries without boroughs using their lat/long coordinates
# This cell maps the coordinates to boroughs using a Point-in-Polygon spatial join with the official NYC borough boundaries
import geopandas as gpd
from shapely.geometry import Point

print("Initiating cleaning process...")

# Ensure target columns are clean, uppercase strings
data['borough'] = data['borough'].astype(str).str.upper().str.strip()
data['on_street_name'] = data['on_street_name'].astype(str).str.upper().str.strip()

# Create a clear mask identifying rows that are actually missing a borough (Python NaNs or strings)
is_missing_borough = data['borough'].isin(['NAN', 'NONE', 'UNSPECIFIED', '', 'BLANK']) | data['borough'].isna()

print(f"Total rows currently missing a borough: {is_missing_borough.sum()}")

# Spatial Cleaning using lat and long coordinates
data['latitude'] = pd.to_numeric(data['latitude'], errors='coerce')
data['longitude'] = pd.to_numeric(data['longitude'], errors='coerce')

has_coords = data['latitude'].notna() & data['longitude'].notna() & (data['latitude'] != 0) & (data['longitude'] != 0)
spatial_target_mask = is_missing_borough & has_coords

data_spatial_ready = data[spatial_target_mask].copy()

if not data_spatial_ready.empty:
    print(f"-> Processing {len(data_spatial_ready)} rows via Point-in-Polygon spatial mapping...")
    
    geometry = [Point(xy) for xy in zip(data_spatial_ready['longitude'], data_spatial_ready['latitude'])]
    gdf_crashes = gpd.GeoDataFrame(data_spatial_ready, geometry=geometry, crs="EPSG:4326")
    
    # GitHub public backup mirror link
    nyc_url = "https://raw.githubusercontent.com/dwillis/nyc-maps/master/boroughs.geojson"
    nyc_boroughs = gpd.read_file(nyc_url).to_crs("EPSG:4326")
    
    joined = gpd.sjoin(gdf_crashes, nyc_boroughs, how="left", predicate="intersects")
    
    # Dynamic column resolver
    actual_boro_col = 'BoroName' if 'BoroName' in joined.columns else 'boro_name'
    data.loc[data_spatial_ready.index, 'borough'] = joined[actual_boro_col].str.upper()

# Re-evaluate which rows are still missing boroughs after the spatial pass
is_missing_borough = data['borough'].isin(['NAN', 'NONE', 'UNSPECIFIED', '', 'BLANK']) | data['borough'].isna()
data_text_ready = data[is_missing_borough].copy()

if not data_text_ready.empty:
    print(f"-> Processing remaining {len(data_text_ready)} rows via text keyword matching...")
    
    # Sequential cascading regex rules to process shared expressways cleanly
    text_rules = {
        r'.*BROOKLYN QUEENS.*': 'BROOKLYN',
        r'.*QUEENSBORO.*': 'QUEENS',
        r'.*BRONX.*': 'BRONX',
        r'.*STATEN ISLAND.*': 'STATEN ISLAND',
        r'.*MANHATTAN.*': 'MANHATTAN',
        r'.*QUEENS.*': 'QUEENS',
        r'.*BROOKLYN.*': 'BROOKLYN'
    }
    
    # Run the text checks down the remaining blank spots
    for pattern, borough_assignment in text_rules.items():
        matched_indices = data_text_ready[data_text_ready['on_street_name'].str.contains(pattern, na=False, regex=True)].index
        
        # Double check to only overwrite if the row wasn't already updated
        current_blanks = data.loc[matched_indices, 'borough'].isin(['NAN', 'NONE', 'UNSPECIFIED', '', 'BLANK']) | data.loc[matched_indices, 'borough'].isna()
        target_indices = matched_indices[current_blanks]
        
        data.loc[target_indices, 'borough'] = borough_assignment

# Standardize any remaining completely unresolvable entries to a uniform 'UNSPECIFIED' string
data['borough'] = data['borough'].replace(['NAN', 'NONE', '', 'BLANK'], 'UNSPECIFIED')
data['borough'] = data['borough'].fillna('UNSPECIFIED')

print("\nNew distribution counts:")
print(data['borough'].value_counts())

In [11]:
# Accident factor formatting
factor_cols = ['contributing_factor_vehicle_1', 'contributing_factor_vehicle_2']

typo_map = {
    'Illnes': 'Illness',
    'Reaction To Other Uninvolved Vehicle': 'Reaction To Uninvolved Vehicle'
}

for col in factor_cols:
    # Type Conversion and stripping
    data[col] = data[col].astype(str).str.strip()
    data[col] = data[col].replace(['nan', 'NaN', 'None', ''], 'Unspecified')

    # Title case
    data[col] = data[col].str.title()

    # Clean typos from map
    data[col] = data[col].replace(typo_map)

    # Wipe out values that have no descriptive val
    data[col] = data[col].apply(lambda x: 'Unspecified' if str(x).isdigit() else x)

In [12]:
# Drop unneeded columns to free memory
columns_to_drop = [
    'location',
    'contributing_factor_vehicle_3', 'contributing_factor_vehicle_4', 'contributing_factor_vehicle_5',
    'vehicle_type_code_3', 'vehicle_type_code_4', 'vehicle_type_code_5'
]
data = data.drop(columns=columns_to_drop)

In [ ]:
# Data sanity checks
print(f"Unique vehicle types: {data['vehicle_type_code_1'].unique()}")
print(f"Unique contributing factors: {data['contributing_factor_vehicle_1'].unique()}")

print(f"Number of persons injured: {data['number_of_persons_injured'].sum()}")
print(f"Number of persons killed: {data['number_of_persons_killed'].sum()}")


In [16]:
# Reorder new columns
cols = list(data.columns)

cols.remove('crash_datetime')
cols.remove('primary_street')

new_col_order = ['crash_datetime', 'primary_street'] + cols

# apply to df
data = data[new_col_order]

data.head(10)

,crash_datetime,primary_street,borough,zip_code,latitude,longitude,on_street_name,cross_street_name,off_street_name,number_of_persons_injured,...,number_of_pedestrians_killed,number_of_cyclist_injured,number_of_cyclist_killed,number_of_motorist_injured,number_of_motorist_killed,contributing_factor_vehicle_1,contributing_factor_vehicle_2,collision_id,vehicle_type_code_1,vehicle_type_code_2
0,2021-09-11 02:39:00,WHITESTONE EXPRESSWAY,NaN,NaN,NaN,NaN,WHITESTONE EXPRESSWAY,20 AVENUE,NaN,2,...,0,0,0,2,0,Aggressive Driving/Road Rage,Unspecified,4455765,PASSENGER CAR,PASSENGER CAR
1,2022-03-26 11:45:00,QUEENSBORO BRIDGE UPPER,NaN,NaN,NaN,NaN,QUEENSBORO BRIDGE UPPER,NaN,NaN,1,...,0,0,0,1,0,Pavement Slippery,NaN,4513547,PASSENGER CAR,NaN
2,2023-11-01 01:29:00,OCEAN PARKWAY,BROOKLYN,11230,40.62,-73.97,OCEAN PARKWAY,AVENUE K,NaN,1,...,0,0,0,1,0,Unspecified,Unspecified,4675373,BICYCLE/SCOOTER,PASSENGER CAR
3,2022-06-29 06:55:00,THROGS NECK BRIDGE,NaN,NaN,NaN,NaN,THROGS NECK BRIDGE,NaN,NaN,0,...,0,0,0,0,0,Following Too Closely,Unspecified,4541903,PASSENGER CAR,COMMERCIAL/TRUCK
4,2022-09-21 13:21:00,BROOKLYN BRIDGE,NaN,NaN,NaN,NaN,BROOKLYN BRIDGE,NaN,NaN,0,...,0,0,0,0,0,Passing Too Closely,Unspecified,4566131,SUV/STATION WAGON,NaN
5,2023-04-26 13:30:00,WEST 54 STREET,NaN,NaN,NaN,NaN,WEST 54 STREET,NaN,NaN,0,...,0,0,0,0,0,Unspecified,Unspecified,4623759,PASSENGER CAR,COMMERCIAL/TRUCK
6,2023-11-01 07:12:00,HUTCHINSON RIVER PARKWAY,NaN,NaN,NaN,NaN,HUTCHINSON RIVER PARKWAY,NaN,NaN,0,...,0,0,0,0,0,Following Too Closely,Driver Inattention/Distraction,4675709,PASSENGER CAR,SUV/STATION WAGON
7,2023-11-01 08:01:00,WEST 35 STREET,NaN,NaN,NaN,NaN,WEST 35 STREET,HENRY HUDSON RIVER,NaN,0,...,0,0,0,0,0,Failure To Yield Right-Of-Way,NaN,4675769,PASSENGER CAR,NaN
8,2023-04-26 22:20:00,61 ED KOCH QUEENSBOROUGH BRIDGE,NaN,NaN,NaN,NaN,NaN,NaN,61 ED KOCH QUEENSBOROUGH BRIDGE,0,...,0,0,0,0,0,Unspecified,NaN,4623865,PASSENGER CAR,COMMERCIAL/TRUCK
9,2021-09-11 09:35:00,1211 LORING AVENUE,BROOKLYN,11208,40.67,-73.87,NaN,NaN,1211 LORING AVENUE,0,...,0,0,0,0,0,Unspecified,NaN,4456314,PASSENGER CAR,NaN


In [22]:
#Export to new csv
data.to_csv(file_path+'cleaned_crash_data.csv', index=False)